In [ ]:
import sys
import time
import random
from PyQt5.QtWidgets import QApplication, QWidget, QVBoxLayout, QLabel, QPushButton
from PyQt5.QtCore import QThread, pyqtSignal, Qt

# --- 1. The VLM Worker Thread ---
class VLMWorker(QThread):
    # Define a signal that will carry a string (str) to the GUI
    # This MUST be defined at the class level
    new_text_received = pyqtSignal(str)

    def run(self):
        """
        This code runs in the BACKGROUND. 
        It simulates your VLM processing data.
        """
        descriptions = [
            "Robot sees: A red ball",
            "Robot sees: A wooden chair",
            "Robot sees: A person waving",
            "Robot sees: An obstacle ahead",
            "Robot sees: Nothing interesting"
        ]

        while True:
            # Simulate waiting for VLM inference (2 seconds)
            time.sleep(2) 
            
            # Pick a random "VLM output"
            simulated_output = random.choice(descriptions)
            
            # CRITICAL: We do NOT update the label here directly.
            # We EMIT the signal containing the data.
            self.new_text_received.emit(simulated_output)

# --- 2. The Main GUI Window ---
class RobotDisplay(QWidget):
    def __init__(self):
        super().__init__()
        
        self.initUI()
        
        # Initialize the VLM Worker
        self.worker = VLMWorker()
        
        # --- CONNECT THE SIGNAL ---
        # "When the worker emits 'new_text_received', run 'self.update_vlm_label'"
        self.worker.new_text_received.connect(self.update_vlm_label)
        
        # Start the background thread
        self.worker.start()

    def initUI(self):
        layout = QVBoxLayout()

        # Label to show the VLM output
        self.status_label = QLabel("Waiting for VLM...", self)
        self.status_label.setAlignment(Qt.AlignCenter)
        # Add some styling so it's easy to see
        self.status_label.setStyleSheet("font-size: 24px; font-weight: bold; color: #333; border: 2px solid gray; padding: 10px;")
        
        # A button to prove the GUI is not frozen
        self.btn = QPushButton("Click me! (I still work)", self)
        self.click_count = 0
        self.btn.clicked.connect(self.count_clicks)

        layout.addWidget(self.status_label)
        layout.addWidget(self.btn)

        self.setLayout(layout)
        self.setWindowTitle('Async VLM Viewer')
        self.resize(400, 200)

    def update_vlm_label(self, text):
        """
        This function (slot) runs in the MAIN THREAD.
        It is safe to update the UI here.
        """
        self.status_label.setText(text)
        print(f"GUI Received: {text}")

    def count_clicks(self):
        self.click_count += 1
        self.btn.setText(f"Clicked {self.click_count} times")

if __name__ == '__main__':
    app = QApplication(sys.argv)
    ex = RobotDisplay()
    ex.show()
    sys.exit(app.exec_())

In [ ]:
import sys
from PyQt5.QtWidgets import QApplication, QWidget, QVBoxLayout, QHBoxLayout, QPushButton, QLabel
from PyQt5.QtCore import pyqtSignal, Qt

class DemoQtWidget(QWidget):
    # 1. Define the Signal
    # This signal will carry the Name (str) and the State (bool)
    module_toggled = pyqtSignal(str, bool)

    def __init__(self):
        super().__init__()
        self.button_list = ['STT', 'TTS', 'VLM', 'UserTracking', 'Patrol', 'Kill', 'Pan-tilt']
        self.state_list = [False] * len(self.button_list) 
        self.state_label_list = []
        # We don't need info_label_list logic for this example, but keeping structure
        self.initUI()

    def initUI(self):
        # Main Horizontal Layout
        main_layout = QHBoxLayout()
        
        # We can remove the main label if you want the buttons to take up space, 
        # or keep it on the far left.
        
        for i, btn_name in enumerate(self.button_list):
            part_layout = QVBoxLayout()
            
            # Button
            btn = QPushButton(btn_name, self)
            # Use lambda to pass the specific name
            btn.clicked.connect(lambda checked, name=btn_name: self.update_state(name))
            part_layout.addWidget(btn)
            
            # State Label (ON/OFF)
            state_label = QLabel("OFF", self)
            state_label.setAlignment(Qt.AlignCenter)
            state_label.setStyleSheet("color: red; font-weight: bold;")
            self.state_label_list.append(state_label)
            part_layout.addWidget(state_label)
            
            # Info Label
            info_label = QLabel(f"Waiting...", self)
            info_label.setAlignment(Qt.AlignCenter)
            part_layout.addWidget(info_label)
            
            main_layout.addLayout(part_layout)

        self.setLayout(main_layout)
        self.setWindowTitle('Robot Control Panel')
        self.resize(600, 150)

    def update_state(self, btn_name):
        # Get index
        index = self.button_list.index(btn_name)
        
        # Toggle internal boolean state
        self.state_list[index] = not self.state_list[index]
        is_active = self.state_list[index]
        
        # Update UI
        lbl = self.state_label_list[index]
        lbl.setText("ON" if is_active else "OFF")
        lbl.setStyleSheet("color: green; font-weight: bold;" if is_active else "color: red; font-weight: bold;")

        # --- 2. EMIT THE SIGNAL TO OUTSIDE ---
        # This tells the Main Robot Controller that something changed
        self.module_toggled.emit(btn_name, is_active)


# --- EXTERNAL LOGIC (The "Outside") ---
class RobotController:
    """
    This class simulates your actual robot logic (clients).
    It doesn't know about PyQt widgets. It just waits for commands.
    """
    def __init__(self):
        self.vlm_client = None # Placeholder
        self.tts_client = None # Placeholder

    def handle_module_change(self, name, is_on):
        """
        This function is called automatically when the GUI signal is emitted.
        """
        print(f"\n[Robot Controller] Received Command: Set {name} to {is_on}")

        if name == "VLM":
            if is_on:
                print(" -> Starting VLM Client connection...")
                # self.vlm_client.connect()
            else:
                print(" -> Stopping VLM Client...")
                # self.vlm_client.disconnect()
        
        elif name == "TTS":
            if is_on:
                print(" -> Unmuting Microphone / Starting TTS...")
            else:
                print(" -> Muting TTS...")
                
        elif name == "Kill":
            if is_on:
                print(" -> !!! EMERGENCY STOP TRIGGERED !!!")


if __name__ == '__main__':
    app = QApplication(sys.argv)
    
    # 1. Create the GUI
    gui = DemoQtWidget()
    
    # 2. Create the External Logic
    robot_brain = RobotController()

    # 3. CONNECT THEM
    # This acts like a bridge. When GUI shouts "module_toggled", 
    # RobotBrain hears it and runs "handle_module_change"
    gui.module_toggled.connect(robot_brain.handle_module_change)

    gui.show()
    sys.exit(app.exec_())